In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import PowerTransformer
from imblearn.over_sampling import SMOTE

from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [2]:
df=pd.read_csv('../data/interim/04-bank_marketing-feature.csv')
df

,age,education,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,contacted_before,campaign_intensity,has_loan_or_housing
0,1.819767,4,-0.822392,0.209868,-0.357749,0.667066,0.772372,0.833632,0.730930,0.348270,...,0,1,0,0,0,1,0,0,0.000000,0
1,-0.208732,4,-0.822392,0.209868,-0.357749,0.667066,0.772372,0.833632,0.730930,0.348270,...,0,1,0,0,0,1,0,0,0.000000,1
2,1.718342,4,-0.822392,0.209868,-0.357749,0.667066,0.772372,0.833632,0.730930,0.348270,...,0,1,0,0,0,1,0,0,0.000000,1
3,2.022617,5,-0.822392,0.209868,-0.357749,0.667066,0.772372,0.833632,0.730930,0.348270,...,0,1,0,0,0,1,0,0,0.000000,0
4,0.196968,6,-0.822392,0.209868,-0.357749,0.667066,0.772372,0.833632,0.730930,0.348270,...,0,1,0,0,0,1,0,0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28653,3.239716,5,-0.822392,0.209868,-0.357749,-0.710192,2.110553,-2.230322,-1.450837,-2.682397,...,0,0,0,0,0,1,0,0,0.000000,1
28654,0.704092,5,-0.822392,0.209868,-0.357749,-0.710192,2.110553,-2.230322,-1.450837,-2.682397,...,0,0,0,0,0,1,0,0,0.000000,0
28655,1.718342,6,-0.179508,0.209868,-0.357749,-0.710192,2.110553,-2.230322,-1.450837,-2.682397,...,0,0,0,0,0,1,0,0,0.000000,1
28656,0.501243,5,-0.822392,0.209868,-0.357749,-0.710192,2.110553,-2.230322,-1.450837,-2.682397,...,0,0,0,0,0,1,0,0,0.000000,0


In [3]:

# 1. Copia
df_fe = df.copy()


# Selección preliminar
X = df_fe.drop(columns=['y'])
y = df_fe['y']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Balanceo solo en train
smote = SMOTE(random_state=42)
X_train_smo, y_train_smo = smote.fit_resample(X_train, y_train)

ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

print("Original train:")
print(y_train.value_counts(normalize=True))

print("RandomOverSampler:")
print(y_train_ros.value_counts())

print("\nRandomUnderSampler:")
print(y_train_rus.value_counts())

print("Smote:")
print(y_train_smo.value_counts(normalize=True))

print("Test original:")
print(y_test.value_counts(normalize=True))

c:\Users\Marcelo\miniforge3\envs\environment\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Original train:
y
0    0.876254
1    0.123746
Name: proportion, dtype: float64
RandomOverSampler:
y
0    20089
1    20089
Name: count, dtype: int64

RandomUnderSampler:
y
0    2837
1    2837
Name: count, dtype: int64
Smote:
y
0    0.5
1    0.5
Name: proportion, dtype: float64
Test original:
y
0    0.876308
1    0.123692
Name: proportion, dtype: float64


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, recall_score

# Definimos los sets a probar
sets_de_entrenamiento = {
    "Original": (X_train, y_train),
    "OverSampler (ROS)": (X_train_ros, y_train_ros),
    "UnderSampler (RUS)": (X_train_rus, y_train_rus),
    "SMOTE": (X_train_smo, y_train_smo)
}

resultados = []

print("--- Evaluación de Modelos ---")

for nombre, (X_tr, y_tr) in sets_de_entrenamiento.items():
    # Entrenamos un modelo rápido
    modelo = RandomForestClassifier(random_state=42, n_estimators=100)
    modelo.fit(X_tr, y_tr)
    
    # Predecimos sobre el set de TEST (que siempre debe ser el original)
    y_pred = modelo.predict(X_test)
    
    # Guardamos métricas
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred) # Vital para ver cuántos 'yes' detectamos
    
    resultados.append({'Técnica': nombre, 'F1-Score': f1, 'Recall': recall})
    
    print(f"\nResultados con {nombre}:")
    print(classification_report(y_test, y_pred))

# Crear tabla comparativa final
df_resultados = pd.DataFrame(resultados).sort_values(by='F1-Score', ascending=False)
print("\n=== COMPARATIVA FINAL ===")
print(df_resultados)

--- Evaluación de Modelos ---

Resultados con Original:
              precision    recall  f1-score   support

           0       0.91      0.96      0.94      5023
           1       0.55      0.32      0.40       709

    accuracy                           0.88      5732
   macro avg       0.73      0.64      0.67      5732
weighted avg       0.86      0.88      0.87      5732


Resultados con OverSampler (ROS):
              precision    recall  f1-score   support

           0       0.92      0.93      0.93      5023
           1       0.47      0.41      0.44       709

    accuracy                           0.87      5732
   macro avg       0.69      0.67      0.68      5732
weighted avg       0.86      0.87      0.87      5732


Resultados con UnderSampler (RUS):
              precision    recall  f1-score   support

           0       0.95      0.78      0.86      5023
           1       0.31      0.68      0.42       709

    accuracy                           0.77      5732
 

In [5]:
# Definimos los sets finales basados en el ganador: SMOTE
X_train_final = X_train_smo
y_train_final = y_train_smo

print("Dataset preparado con SMOTE.")
print(f"Registros totales para entrenamiento: {len(X_train_final)}")
print(f"Distribución de clases: \n{y_train_final.value_counts(normalize=True)}")

Dataset preparado con SMOTE.
Registros totales para entrenamiento: 40178
Distribución de clases: 
y
0    0.5
1    0.5
Name: proportion, dtype: float64


In [6]:
# Guardar el dataset balanceado  para el modelo
target='y'
train_balanced = X_train_final.copy()
train_balanced[target] = y_train_final.values

test_original = X_test.copy()
test_original[target] = y_test.values

train_balanced.to_csv("../data/processed/train_balanced.csv", index=False)
test_original.to_csv("../data/processed/test_original.csv", index=False)

print("Archivos guardados en ../data/processed/")

Archivos guardados en ../data/processed/


In [7]:
train_balanced

,age,education,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,contacted_before,campaign_intensity,has_loan_or_housing,y
0,1.819767,4,-0.822392,0.209868,-0.357749,0.667066,0.772372,0.833632,0.730930,0.348270,...,1,0,0,0,1,0,0,0.000000,1,0
1,-1.020131,6,-0.822392,0.209868,-0.357749,-1.148410,-1.133627,-1.251559,-1.270780,-0.876524,...,0,0,1,0,1,0,0,0.000000,0,0
2,-0.310157,6,-0.822392,0.209868,-0.357749,0.667066,0.772372,0.833632,0.732069,0.348270,...,0,0,0,1,1,0,0,0.000000,0,0
3,-0.614432,6,2.392025,0.209868,-0.357749,-1.148410,-1.220185,-2.060102,-1.154541,-0.876524,...,0,0,0,1,1,0,0,3.724439,1,1
4,-1.222981,6,-0.822392,0.209868,-0.357749,-1.837038,-2.331585,1.897505,-1.532889,-1.181723,...,0,0,0,1,1,0,0,0.000000,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40173,-1.473090,6,-0.822392,0.209868,-0.357749,-1.837038,-1.012446,-0.102576,-1.317610,-1.181723,...,0,0,0,0,1,0,0,0.000000,0,1
40174,-1.366904,5,-0.361960,-4.769067,3.471334,-2.150051,-2.023440,2.237944,-1.600272,-1.964046,...,0,0,1,0,0,0,1,0.000000,1,1
40175,-1.731495,4,-0.822392,0.209868,5.385876,-1.148410,0.161573,1.204551,-1.670041,-2.081328,...,1,0,0,0,0,0,1,0.000000,0,1
40176,-0.554071,6,-0.822392,0.209868,-0.357749,-2.150051,-1.933420,2.854990,-1.625196,-1.964046,...,0,1,0,0,1,0,0,0.000000,0,1


In [8]:
test_original

,age,education,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,contacted_before,campaign_intensity,has_loan_or_housing,y
19889,0.298393,4,-0.822392,0.209868,-0.357749,-1.148410,-0.818557,-1.443056,-1.236023,-0.876524,...,0,0,0,0,1,0,0,0.000000,1,0
27935,-0.817282,6,-0.179508,0.209868,-0.357749,-1.085807,0.829500,0.429360,-1.524912,-2.309228,...,1,0,0,0,1,0,0,0.000000,0,0
28203,-0.107307,6,-0.179508,0.209868,1.556793,-0.710192,1.127259,0.599580,-1.535168,-2.682397,...,0,0,0,1,0,0,1,0.000000,0,0
22457,-0.715857,4,-0.179508,0.209868,-0.357749,-1.148410,-1.133627,-1.251559,-1.296421,-0.876524,...,1,0,0,0,1,0,0,0.000000,1,0
26569,1.718342,4,-0.822392,0.209868,-0.357749,-2.150051,-1.556028,2.174111,-1.626336,-1.964046,...,0,0,1,0,1,0,0,0.000000,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3629,1.008367,4,1.106258,0.209868,-0.357749,0.667066,0.772372,0.833632,0.730930,0.348270,...,1,0,0,0,1,0,0,1.722470,0,0
10228,-0.310157,6,0.463375,0.209868,-0.357749,0.854874,0.640805,-0.506848,0.791329,0.842720,...,0,0,0,1,1,0,0,0.721486,1,0
6605,-1.020131,4,-0.179508,0.209868,-0.357749,0.854874,1.587745,-0.315351,0.790189,0.842720,...,0,0,1,0,1,0,0,0.000000,1,0
6461,-0.208732,5,-0.822392,0.209868,-0.357749,0.854874,1.587745,-0.315351,0.790189,0.842720,...,0,0,1,0,1,0,0,0.000000,1,0
